"""
# Bonsai 2 on TPU: fixed-KV, batched (notebook 06)

Notebook 04 proved correctness (top-5 exact vs llama.cpp) at 0.27 tok/s.
This notebook attacks the actual bottleneck found in notebook 05's profile:
the run was **recompile-bound, not bandwidth-bound**.

Notebook 04's `attn_layer` grows its KV cache with `jnp.concatenate`:

    K = jnp.concatenate([kc, k[None]], 0)

`kc.shape[0]` is a Python int that changes every step, so all 16 attention
layers get a **fresh XLA compilation on every token**. That is why a single
matvec showed ~5 GB/s effective against ~800 GB/s of HBM peak: the device
was mostly waiting on the compiler.

Three changes, each with a defined purpose:

1. **Fixed-size KV + `dynamic_update_slice`.** The cache is preallocated to
   `MAX_T` and written with a *traced* `pos`, so one compiled program serves
   every position. A causal mask (`arange(MAX_T) <= pos`) preserves exact
   attention semantics. No shape ever changes -> no recompile.
2. **`jax.vmap` over the batch.** The proven single-row math from notebook
   04 is lifted, not reimplemented, so correctness carries over. Weights
   decode once per layer and are shared, so `[B,N] @ [N,M]` keeps the MXU
   busy instead of running a lonely matvec.
3. **Per-layer jitted units.** Each GDN / attention / FFN is one compiled
   program, cutting per-token dispatches from ~1000 to ~200.

A correctness gate runs first: if the fast path diverges from notebook 04 it
fails loudly instead of silently generating garbage.
"""


In [ ]:
import json
import os
import time

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")

import jax
import jax.numpy as jnp
import numpy as np

print(f"jax {jax.__version__} | backend {jax.default_backend()}")

GGUF = os.environ.get(
    "GGUF_PATH", "/content/gguf/Ternary-Bonsai-2-27B-PTQ1_0.gguf")
N_GEN = int(os.environ.get("N_TOKENS", "16"))
PROMPT = os.environ.get("PROMPT", "Hello")
BATCH = int(os.environ.get("BATCH", "16"))
MAX_T = int(os.environ.get("MAX_T", "256"))


In [ ]:
_src = open(os.environ.get("FWD_SRC", "/content/04.py"),
            encoding="utf-8").read()
fw = {"__name__": "fwd"}
exec(compile(_src.split("st = {")[0], "fwd.py", "exec"), fw)

cfg = fw["cfg"]
NL, IS_RECR = cfg["n_layer"], tuple(cfg["is_recr"])
EPS, NEMB = cfg["eps"], cfg["n_embd"]
H, HKV, HD = cfg["n_head"], cfg["n_head_kv"], cfg["head_dim"]
ST, NV, NK, REP = cfg["state"], cfg["n_v"], cfg["n_k"], cfg["rep"]
KEY_DIM, CONV, CONV_DIM = cfg["key_dim"], cfg["conv"], cfg["conv_dim"]
NFF = cfg["n_ff"]

rms, rope, tok = fw["rms"], fw["rope"], fw["tok"]
l2n, fold = fw["l2n"], fw["fold"]
decode_layer, SMALLW = fw["decode_layer"], fw["SMALLW"]
embed, O_NORM = fw["embed"], fw["O_NORM"]
chunked_logits = fw["chunked_logits"]
SIGNS, ROW = fw["SIGNS"], fw["SIGNS"][NEMB]

_idxtri = jnp.arange(NV) % NK           # cyclic head routing (h % n_k)
_hk = jnp.arange(NK)[:, None]
_r = jnp.arange(REP)[None, :]
_perm = _hk + _r * NK                    # ssm_out permute [st, nk, rep]
_kidx = jnp.arange(H) // (H // HKV)      # GQA head mapping
_isscale = 1.0 / jnp.sqrt(jnp.float32(HD))
_sscale = 1.0 / jnp.sqrt(jnp.float32(ST))


"""
## The three fused layer units

`gdn_unit`, `attn_unit` and `ffn_unit` are the only new math here. Each is
`jax.jit`-compiled; `attn_unit` takes `pos` as a **traced** scalar, which is
what removes the per-step recompilation.
"""


In [ ]:
@jax.jit
def gdn_unit(x, W, S, SW, conv_hist):
    """x [n_embd] -> (attn_out, new_state, new_hist).

    Gated delta net. Every rule below is ported from notebook 04, which
    verified each against `models/qwen35.cpp` and `ops.cpp`:
      - q/k/v are contiguous 128-dim head blocks; the [head_k_dim, 2*n_k]
        view is Fortran, so the reshape must transpose
      - head routing is cyclic (h % n_k), not block (h // rep)
      - ssm_a already holds -exp(A_log); never exponentiate twice
      - gated norm is silu(z) * rms_norm(o), i.e. sigmoid(z) * z * o
    """
    xf = fold(x, ROW)
    qkv = xf @ W["attn_qkv.weight"]                    # [2*key_dim + val_dim]
    z = xf @ W["attn_gate.weight"]                     # [val_dim]
    beta = jax.nn.sigmoid(x @ SW["ssm_beta.weight"])
    alpha = x @ SW["ssm_alpha.weight"]
    gate = SW["ssm_a"] * jnp.logaddexp(alpha + SW["ssm_dt.bias"], 0.0)

    win = jnp.concatenate([conv_hist, qkv[None, :]], 0)
    y = jnp.sum(win * SW["ssm_conv1d.weight"], 0)      # causal conv
    y = y * jax.nn.sigmoid(y)                          # SiLU

    qk = l2n(y[:2 * KEY_DIM].reshape(2 * NK, ST).T, EPS)
    q, k = qk[:, :NK], qk[:, NK:]                      # [st, n_k]
    v = y[2 * KEY_DIM:].reshape(NV, ST).T              # [st, n_v]
    q, k = q[:, _idxtri], k[:, _idxtri]                # -> [st, n_v]

    Sg = S * jnp.exp(gate)[None, None, :]
    sk = jnp.einsum("ijv,iv->jv", Sg, k)
    dv = (v - sk) * beta[None, :]
    Sg = Sg + jnp.einsum("iv,jv->ijv", k, dv)
    o = jnp.einsum("ijv,iv->jv", Sg, q) * _sscale

    # gated norm (qwen35.cpp:329-332): silu(z) * rms_norm(o, ssm_norm)
    zz = z.reshape(NV, ST).T
    o = o * jax.lax.rsqrt(jnp.mean(o * o, axis=0, keepdims=True) + EPS)
    o = o * SW["ssm_norm.weight"][:, None]
    o = jax.nn.sigmoid(zz) * zz * o

    Q = jnp.transpose(o[:, _perm], (1, 2, 0))         # [nk, rep, st]
    out = fold(Q.reshape(-1), SIGNS[ST * NV]) @ W["ssm_out.weight"]
    return out, Sg, win[1:, :]


@jax.jit
def attn_unit(x, W, SW, K, V, pos):
    """x [n_embd] -> (attn_out, K, V) with a fixed [MAX_T, HKV, HD] cache.

    `pos` is traced: `dynamic_update_slice` writes position `pos` and the
    mask admits keys <= pos, so one compiled program serves every step.
    """
    xf = fold(x, ROW)
    qg = (xf @ W["attn_q.weight"]).reshape(H, 2, HD)
    q, g = qg[:, 0, :], qg[:, 1, :]
    q = rms(q, SW["attn_q_norm.weight"], EPS)
    k = rms((xf @ W["attn_k.weight"]).reshape(HKV, HD),
            SW["attn_k_norm.weight"], EPS)
    v = (xf @ W["attn_v.weight"]).reshape(HKV, HD)
    q, k = rope(q, pos), rope(k, pos)

    K = jax.lax.dynamic_update_slice(K, k[None, None, :, :], (pos, 0, 0, 0))
    V = jax.lax.dynamic_update_slice(V, v[None, None, :, :], (pos, 0, 0, 0))

    scores = jnp.einsum("hd,thd->ht", q, K[:, _kidx, :]) * _isscale
    allow = jnp.arange(K.shape[0]) <= pos
    scores = jnp.where(allow[None, :], scores, -1e30)
    p = jax.nn.softmax(scores, -1)
    o = jnp.einsum("ht,thd->hd", p, V[:, _kidx, :])
    o = o * jax.nn.sigmoid(g)
    return fold(o.reshape(-1), SIGNS[H * HD]) @ W["attn_output.weight"], K, V


@jax.jit
def ffn_unit(x, W):
    xf = fold(x, ROW)
    g = xf @ W["ffn_gate.weight"]
    u = xf @ W["ffn_up.weight"]
    return fold(jax.nn.silu(g) * u, SIGNS[NFF]) @ W["ffn_down.weight"]


gdn_batch = jax.vmap(gdn_unit, in_axes=(0, None, 0, None, 0))
attn_batch = jax.vmap(attn_unit, in_axes=(0, None, None, 0, 0, 0))
ffn_batch = jax.vmap(ffn_unit, in_axes=(0, None))
logits_batch = jax.vmap(lambda h: chunked_logits(rms(h, O_NORM, EPS)))


"""
## Correctness gate: fused units vs notebook 04's proven path

Compares the new fused GDN unit against notebook 04's reference on identical
input, on layer 0, before any timing. Also checks that batched rows agree.
"""


In [ ]:
x0 = embed(int(tok.encode(PROMPT)[0]))
W0, SW0 = decode_layer(0), SMALLW[0]
h0 = rms(x0, SW0["attn_norm.weight"], EPS)
S0 = jnp.zeros([ST, ST, NV], jnp.float32)
C0 = jnp.zeros([CONV - 1, CONV_DIM], jnp.float32)

a_ref, _, _ = fw["gdn_layer"](h0, W0, S0, SW0, C0)
a_new, _, _ = gdn_unit(h0, W0, S0, SW0, C0)
d_single = float(jnp.abs(a_new - a_ref).max())
a_v, _, _ = gdn_batch(jnp.stack([h0, h0]), W0,
                      jnp.stack([S0, S0]), SW0, jnp.stack([C0, C0]))
d_rows = float(jnp.abs(a_v[0] - a_v[1]).max())
print(f"gdn_unit vs notebook 04: {d_single:.3e}  (expect ~0)")
print(f"vmap rows identical:     {d_rows:.3e}")
assert d_single < 1e-4, "fused GDN diverged from the proven path"
assert d_rows == 0.0, "batched rows disagree"
print("gate PASS")


In [ ]:
ids = tok.encode(PROMPT)
B = BATCH
print(f"prompt {PROMPT!r} -> {len(ids)} tokens | batch {B} | MAX_T {MAX_T}")

S = [jnp.zeros([B, ST, ST, NV], jnp.float32) if IS_RECR[i] else None
     for i in range(NL)]
C = [jnp.zeros([B, CONV - 1, CONV_DIM], jnp.float32) if IS_RECR[i] else None
     for i in range(NL)]
K = [None if IS_RECR[i] else jnp.zeros([B, MAX_T, HKV, HD], jnp.float32)
     for i in range(NL)]
V = [None if IS_RECR[i] else jnp.zeros([B, MAX_T, HKV, HD], jnp.float32)
     for i in range(NL)]


def step(x, pos):
    for il in range(NL):
        W, SW = decode_layer(il), SMALLW[il]
        h = rms(x, SW["attn_norm.weight"], EPS)
        if IS_RECR[il]:
            a, S[il], C[il] = gdn_batch(h, W, S[il], SW, C[il])
        else:
            a, K[il], V[il] = attn_batch(h, W, SW, K[il], V[il], pos)
        x = x + a
        x = x + ffn_batch(rms(x, SW["post_attention_norm.weight"], EPS), W)
        del W
    return x


x = jnp.broadcast_to(embed(int(ids[0])), (B, NEMB))


"""
## Warmup (compile, excluded) then the measured decode loop

`pos` advances by a traced increment, so the loop reuses the compiled
programs instead of recompiling per token.
"""


In [ ]:
t0 = time.perf_counter()
x = step(x, jnp.int32(0))
first = np.asarray(jnp.argmax(logits_batch(x), axis=-1))
jax.block_until_ready(x)
t_compile = time.perf_counter() - t0
print(f"compile+prefill: {t_compile:.1f}s (excluded from tok/s)")
print(f"first token: {int(first[0])} {tok.decode([int(first[0])])!r}")


In [ ]:
gen = [[] for _ in range(B)]
cur = first
pos = jnp.int32(1)
t0 = time.perf_counter()
for i_step in range(N_GEN):
    for b in range(B):
        if cur[b] != tok.eos:
            gen[b].append(int(cur[b]))
    x = jax.vmap(embed)(cur)
    x = step(x, pos)
    cur = np.asarray(jnp.argmax(logits_batch(x), axis=-1))
    pos = pos + 1
    if i_step % 4 == 0:
        print(f"  step {i_step}: {tok.decode([int(cur[0])])!r}")
t_gen = time.perf_counter() - t0

n = sum(len(g) for g in gen)
agg = B * N_GEN / t_gen
print("\n=== notebook 06 receipt ===")
print(f"batch            : {B} (one prompt replicated)")
print(f"tokens           : {n} of {B * N_GEN} (eos-truncated)")
print(f"decode wall time : {t_gen:.2f}s")
print(f"aggregate tok/s  : {agg:.1f}")
print(f"per-seq latency  : {t_gen / max(N_GEN, 1) * 1000:.0f} ms/token")
print(f"baseline (nb 04) : 0.27 tok/s single-stream")
print(f"sample: {tok.decode(gen[0][:48])!r}")

receipt = {
    "notebook": "06-bonsai-tpu-fast",
    "backend": str(jax.default_backend()),
    "jax": jax.__version__,
    "batch": B,
    "max_t": MAX_T,
    "prompt": PROMPT,
    "tokens_generated": n,
    "tokens_possible": B * N_GEN,
    "decode_seconds": round(t_gen, 3),
    "aggregate_tok_per_s": round(agg, 2),
    "per_sequence_ms_per_token": round(t_gen / max(N_GEN, 1) * 1000, 1),
    "baseline_notebook04_tok_per_s": 0.27,
    "correctness_gate_pass": True,
    "claim": "[exec] measured on Colab TPU v5e-1; batch replicates one prompt",
}
path = os.environ.get("RECEIPT", "/content/receipt-colab-tpu-06.json")
json.dump(receipt, open(path, "w"), indent=2)
print(f"receipt -> {path}")
